In [ ]:
#Insurance Inference: Algunos tópicos de inferencia con python y nuestra base de datos insurance

In [2]:
pip install altair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.5/797.5 kB 2.7 MB/s  0:00:016m0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import plotly.graph_objects as go
import random
import altair as alt
alt.data_transformers.enable("vegafusion")
from scipy.stats import t
from statistics import variance
rng = np.random.default_rng()
x = stats.uniform.rvs(size=75, random_state=rng)
sns.set(style="darkgrid")

In [2]:
df = pd.read_csv('insurance.csv')

In [3]:
df.shape

(1338, 7)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [5]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
#Muestreo. Distribuciones de muestreo comunes en estadística
#Muestra de una variable continua

np.random.seed(123)

df.sample(n=20)["bmi"]


650     42.680
319     37.335
314     31.400
150     24.130
336     25.740
970     28.160
169     18.905
684     18.500
1097    33.770
512     22.420
597     33.250
698     33.725
630     36.100
738     31.730
696     32.300
827     28.025
18      40.300
31      26.315
682     35.300
185     41.895
Name: bmi, dtype: float64

In [7]:
np.random.seed(123)

df.sample(n=20)["bmi"].describe()

count    20.00000
mean     31.09900
std       7.02806
min      18.50000
25%      26.17125
50%      32.01500
75%      35.50000
max      42.68000
Name: bmi, dtype: float64

In [8]:
#De una variable discreta
np.random.seed(123)
df.sample(n=40)["sex"].value_counts()

sex
male      24
female    16
Name: count, dtype: int64

In [ ]:
#Proporciones
np.random.seed(123)
df.sample(n=40)["sex"].value_counts(normalize=True)

In [ ]:
np.random.seed(123)
df.sample(n=100)["region"].value_counts()

In [ ]:
np.random.seed(123)
df.sample(n=100)["region"].value_counts(normalize=True)

In [9]:
#Una muestra de tamaño 20 de toda la base
np.random.seed(123)
samples = pd.concat([
    df.sample(20).assign(replicate=n)
    for n in range(1)
])
samples

,age,sex,bmi,children,smoker,region,charges,replicate
650,49,female,42.680,2,no,southeast,9800.88820,0
319,32,male,37.335,1,no,northeast,4667.60765,0
314,27,female,31.400,0,yes,southwest,34838.87300,0
150,35,male,24.130,1,no,northwest,5125.21570,0
336,60,male,25.740,0,no,southeast,12142.57860,0
970,50,female,28.160,3,no,southeast,10702.64240,0
169,27,male,18.905,3,no,northeast,4827.90495,0
684,33,female,18.500,1,no,southwest,4766.02200,0
1097,22,male,33.770,0,no,southeast,1674.63230,0
512,51,male,22.420,0,no,northeast,9361.32680,0


In [10]:
#Muchas muestras: 500 muestras de tamaño 20
np.random.seed(123)
samples500 = pd.concat([
    df.sample(20).assign(replicate=n)
    for n in range(500)
])
samples500


,age,sex,bmi,children,smoker,region,charges,replicate
650,49,female,42.680,2,no,southeast,9800.88820,0
319,32,male,37.335,1,no,northeast,4667.60765,0
314,27,female,31.400,0,yes,southwest,34838.87300,0
150,35,male,24.130,1,no,northwest,5125.21570,0
336,60,male,25.740,0,no,southeast,12142.57860,0
...,...,...,...,...,...,...,...,...
1077,21,male,26.030,0,no,northeast,2102.26470,499
1236,63,female,21.660,0,no,northeast,14449.85440,499
182,22,male,19.950,3,no,northeast,4005.42250,499
19,30,male,35.300,0,yes,southwest,36837.46700,499


In [ ]:
#Distribución de muestreo sobre la proporción de sexo (sex)
# saca proporciones
(
    samples500
    .groupby("replicate")
    ["sex"]
    .value_counts(normalize=True) 
)


In [ ]:
(
    samples500
    .groupby("replicate")
    ["sex"]
    .value_counts(normalize=True)
    .reset_index(name="Proporción muestral")
)

In [ ]:
sample_est = (
    samples500
    .groupby("replicate")
    ["sex"]
    .value_counts(normalize=True)
    .reset_index(name="Proporción muestral")
)
sample_est = sample_est[sample_est["sex"] == "female"]
sample_est

In [ ]:
sampling_dist = alt.Chart(sample_est).mark_bar().encode(
    x=alt.X("Proporción muestral")
        .bin(maxbins=30)
        .title("Sexo femenino: Proporciones muestrales"),
    y=alt.Y("count()").title("Count"),
)

sampling_dist

In [ ]:
#El estimador puntual de esta proporción estas muestras
sample_est["Proporción muestral"].mean()

In [ ]:
df.sex.value_counts(normalize=True)

In [ ]:
#Intervalo de confianza
data=pd.DataFrame(sample_est["Proporción muestral"])
confidence = 0.95
df = len(data) -1 #grados de libeta'
mean = np.mean(data)
std_err = np.std(data, ddof=1) / np.sqrt(len(data)) 

# Intervalo de confianza
ci = t.interval(confidence, df, loc=mean, scale=std_err)
print("95% Confidence Interval:", ci)


In [81]:
#Distribución muestral para variables continuas
#Distribución de la variable original
#Algo pasó y tuve que volver a cargar la base
df = pd.read_csv('C:/Users/Salvador/Desktop/Cursos 2026-II/Machine Learning en Seguros/insurance.csv')

In [ ]:
charges_distribution = alt.Chart(df).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=40)
        .title("Charges"),
    y=alt.Y("count()", title="Count"),
)

charges_distribution

In [ ]:
df["charges"].mean()

In [3]:
#Una muestra de toda la base
np.random.seed(123)
df_one_sample = df.sample(n=100)

In [ ]:
sample_distribution = alt.Chart(df_one_sample).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=40)
        .title("Charges"),
    y=alt.Y("count()").title("Count"),
)

sample_distribution

In [ ]:
df_one_sample["charges"].mean()

In [ ]:
#Distribución muestral charges
charges_estimates = (
    samples500
    .groupby("replicate")
    ["charges"]
    .mean()
    .reset_index()
    .rename(columns={"charges": "mean_charges"})
)
charges_estimates

In [ ]:
charges_distribution = alt.Chart(charges_estimates).mark_bar().encode(
    x=alt.X("mean_charges")
        .bin(maxbins=30)
        .title("Media muestral charges"),
    y=alt.Y("count()").title("Count")
)

charges_distribution

In [ ]:
charges_estimates["mean_charges"].mean()

In [ ]:
#Intervalo de confianza
data1=pd.DataFrame(charges_estimates["mean_charges"])
confidence = 0.95
df = len(data1) -1 
mean = np.mean(data1)
std_err = np.std(data1, ddof=1) / np.sqrt(len(data1)) 

# Intervalo de confianza
ci = t.interval(confidence, df, loc=mean, scale=std_err)
print("95% Confidence Interval:", ci)

In [ ]:
#Distribuciones de muestreo: Varianza muestral 
charges_estimates1 = (
    samples500
    .groupby("replicate")
    ["charges"]
    .var()
    .reset_index()
    .rename(columns={"charges": "var_charges"})
)
charges_estimates1

In [ ]:
charges_var_distribution = alt.Chart(charges_estimates1).mark_bar().encode(
    x=alt.X("var_charges")
        .bin(maxbins=30)
        .title("Varianza muestral charges"),
    y=alt.Y("count()").title("Count")
)

charges_var_distribution

In [ ]:
#
charges_estimates1["var_charges"].mean()

In [ ]:
 #Intervalo de confianza
data1=pd.DataFrame(charges_estimates1["var_charges"])
alpha = 0.05
n = len(data1)
var = np.mean(data1)
chi2_lower = stats.chi2.ppf(alpha / 2, df=n - 1)
chi2_upper = stats.chi2.ppf(1 - alpha / 2, df=n - 1) 
ci_lower = (n - 1)*var/chi2_upper
ci_upper = (n - 1)*var/chi2_lower
print(f"95% Intervalo de confianza de la varianza: ({ci_lower:.4f}, {ci_upper:.4f})")

In [ ]:
#Bootstrap. Asumamos que nuestros datos son los que se encuentran en df_one_sample

df_one_sample


In [ ]:
one_sample_dist = alt.Chart(df_one_sample).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=30)
        .title("Charges"),
    y=alt.Y("count()").title("Count"),
)

one_sample_dist

In [ ]:
#Bootstrap con esta variable charges. Una sola remuestra

np.random.seed(123)
boot_charges = df_one_sample.sample(frac=1, replace=True)
boot_charges_dist = alt.Chart(boot_charges).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=30)
        .title("Boot charges"),
    y=alt.Y("count()", title="Count"),
)

boot_charges

In [ ]:
boot_charges_dist

In [ ]:
boot_charges["charges"].mean()

In [ ]:
#Muchas remuestras
boot_charges_10000 = pd.concat([
    df_one_sample.sample(frac=1, replace=True).assign(replicate=n)
    for n in range(10_000)
])
boot_charges_10000

In [ ]:
#Algunos histogramas de estas muestras bootstrap
four_bootstrap_samples = boot_charges_10000.query("replicate < 4")
four_bootstrap_fig = alt.Chart(four_bootstrap_samples, height=150).mark_bar().encode(
    x=alt.X("charges")
        .bin(maxbins=20)
        .title("Charges"),
    y=alt.Y("count()").title("Count")
).facet(
    "replicate:N",
    columns=2
)
four_bootstrap_fig


In [ ]:
#Media de cada una de estas primeras 4 remuestra (replicate)
(
    four_bootstrap_samples
    .groupby("replicate")
    ["charges"]
    .mean()
    .reset_index()
    .rename(columns={"charges": "mean_charges"})
)

In [ ]:
#Media de todas las 10000 remuestras (replicate)
boot_charges_10000_means = (
    boot_charges_10000
    .groupby("replicate")
    ["charges"]
    .mean()
    .reset_index()
    .rename(columns={"charges": "mean_charges"})
)

boot_charges_10000_means

In [ ]:
pip install "vl-convert-python>=1.6.0"

In [ ]:
boot_charges_dist = alt.Chart(boot_charges_10000_means).mark_bar().encode(
    x=alt.X("mean_charges")
        .bin(maxbins=20)
        .title("Bootstrap media charges"),
    y=alt.Y("count()").title("Count"),
)

boot_charges_dist

In [ ]:
boot_charges_10000_means["mean_charges"].mean()

In [ ]:
#Intervalo de confianza empírico (Este intervalo es un intervalo los percentiles de la distribución empírica)
ci_boot_charges = boot_charges_10000_means["mean_charges"].quantile([0.025, 0.975])
ci_boot_charges